In [1]:
import pandas as pd
import requests, zipfile, io, datetime as dt, sys, os
from typing import Tuple
import nse_workday

# -------------------------- USER SETTINGS -----------------------------
CHARTINK_CSV = "chartink_orb.csv"
OUTPUT_CSV   = "enriched_shortlist.csv"
NIFTY_CSV = "nifty.csv"
BHAVCOPY_CSV = "bhavcopy.csv"
# ----------------------------------------------------------------------

NSE_URL_TEMPLATE = (
    "https://nsearchives.nseindia.com/content/cm/BhavCopy_NSE_CM_0_0_0_"
    "{year}{mon}{day}_F_0000.csv.zip"
)

def get_holidays_list(date: dt.date) -> bool:
    # Get the current date and time
    current_year = dt.datetime.now().year
    # Get holidays in a specific date range
    start_date = f"01-01-{current_year}"  # Format: DD-MM-YYYY
    end_date = f"31-12-{current_year}"
    holidays = nse_workday.get_holidays_list(start_date=start_date, end_date=end_date)
    
    # Alternatively, check if a specific date is a holiday
    date_to_check = date  # e.g., Independence Day
    is_holiday = nse_workday.isHoliday(date_to_check)
    return bool(is_holiday)

def next_trading_date(date: dt.date, step: int = -1) -> dt.date:
    """Return the previous(-1) or next(+1) weekday."""
    while True:
        date += dt.timedelta(days=step)
        if date.weekday() < 5:  # Mon-Fri
            return date

def download_and_load_fo(date: dt.date) -> pd.DataFrame:
    """Download NSE FO bhav-copy for a date and return DF or raise."""
 # Format the date
    date_str = date.strftime('%Y%m%d')  # YYYYMMDD
    filename = f'BhavCopy_NSE_FO_0_0_0_{date_str}_F_0000.csv.zip'
    url = f'https://nsearchives.nseindia.com/content/fo/{filename}'

    print(f"Fetching {url}")
    
    # Use headers to simulate a browser
    headers = {
        'User-Agent': 'Mozilla/5.0',
        'Referer': 'https://www.nseindia.com/'
    }

    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code == 200:
            z = zipfile.ZipFile(io.BytesIO(response.content))
            csv_name = z.namelist()[0]
            with z.open(csv_name) as f:
                df = pd.read_csv(f)

            df.rename(columns=
                                {'FinInstrmTp': 'INSTRUMENT'
                                 , 'TckrSymb': 'SYMBOL'
                                 , 'XpryDt': 'EXPIRY_DT'
                                 , 'StrkPric': 'STRIKE_PR'
                                 , 'OptnTp': 'OPTION_TYP'
                                 , 'OpnPric': 'OPEN'
                                 , 'HghPric': 'HIGH'
                                 , 'LwPric': 'LOW'
                                 , 'ClsPric': 'CLOSE'
                                 , 'LastPric': 'SETTLE_PR'
                                 , 'TtlTradgVol': 'CONTRACTS'
                                 , 'TtlTrfVal': 'VAL_INLAKH'
                                 , 'OpnIntrst': 'OPEN_INT'
                                 , 'ChngInOpnIntrst': 'CHG_IN_OI'
                                 , 'TradDt': 'TIMESTAMP'
                                 }, inplace=True)    
            # Standardise column names
            df.columns = df.columns.str.lower()
            # df.to_csv(NIFTY_CSV, index=False, mode='w', header=True)            
            
            return df
        else:
            print(f"Failed to download. HTTP Status Code: {response.status_code}")
            print("Possible reasons: Non-trading day, wrong URL format, or file not available yet.")
            return pd.DataFrame()
    except Exception as e:
        print(f"Error occurred: {e}")
        return pd.DataFrame()

def get_two_bhavcopies() -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Return (today_df, prev_df). If today's file missing, exit."""
    today = dt.date.today()
    # Date Change applies here
    # today = dt.date(2025, 10, 1)
    max_attempts = 7
    attempts = 0
    while attempts < max_attempts:
        try:
            isHoliday = get_holidays_list(today)
            if isHoliday:
                print(f"Skipping holiday: {today}")
                today = next_trading_date(today, step=-1)
                continue

            df_today = download_and_load_fo(today)
            df_today.to_csv(BHAVCOPY_CSV, index=False, mode='w')
            break
        except Exception as e:
            attempts += 1
            today = next_trading_date(today, step=-1)
    else:
        print("Cannot find any bhavcopy for the last 7 calendar days.")
        sys.exit(1)

    prev_day = next_trading_date(today, step=-1)
    attempts = 0
    while attempts < max_attempts:
        try:
            isHoliday = get_holidays_list(prev_day)
            if isHoliday:
                print(f"Skipping holiday: {prev_day}")
                prev_day = next_trading_date(prev_day, step=-1)
                continue

            df_prev = download_and_load_fo(prev_day)
            break
        except Exception as e:
            attempts += 1
            prev_day = next_trading_date(prev_day, step=-1)
    else:
        print("Cannot find previous trading day's bhavcopy.")
        sys.exit(1)

    return df_today, df_prev

def compute_oi_change(df_today: pd.DataFrame, df_prev: pd.DataFrame) -> pd.DataFrame:
    """Merge today's & prev futures OI, compute % change."""
    flds = ["instrument", "symbol", "close", "expiry_dt", "open_int"]
    fut_today = df_today.query("instrument == 'STF'")[flds]
    fut_prev  = df_prev .query("instrument == 'STF'")[["symbol", "open_int"]]
    fut = fut_today.merge(fut_prev, on="symbol", how="left", suffixes=("", "_prev"))
    fut["oi_pct_chg"] = 100 * (fut["open_int"] - fut["open_int_prev"]) / fut["open_int_prev"]
    # Keep latest expiry per symbol
    fut_latest = fut.sort_values("expiry_dt").drop_duplicates("symbol", keep="first")
    return fut_latest[["symbol", "oi_pct_chg", "close"]]

def compute_pcr(df_today: pd.DataFrame) -> pd.DataFrame:
    """Compute Put-Call Ratio per underlying from today's options."""
    opt = df_today.query("instrument == 'STO'")
    grouped = opt.groupby(["symbol", "option_typ"])["open_int"].sum().unstack(fill_value=0)
    grouped["pcr"] = grouped.get("PE", 0) / grouped.get("CE", 1)
    grouped = grouped.reset_index()[["symbol", "pcr"]]
    return grouped

def main():

    if not os.path.exists(CHARTINK_CSV):
        print(f"Cannot locate {CHARTINK_CSV}")
        sys.exit(1)

    # 1. Load Chartink shortlist
    shortlist = pd.read_csv(CHARTINK_CSV)
    shortlist["symbol"] = shortlist["symbol"].str.upper()
    # Date Change applies here
    # dt.date(2025, 8, 8)
    shortlist["Date"] = dt.date.today()


    # 2. Bhav-copy data
    df_today, df_prev = get_two_bhavcopies()

    # 3. Enrich with OI change
    oi_df  = compute_oi_change(df_today, df_prev)

    pcr_df = compute_pcr(df_today)

    df = shortlist.merge(oi_df[["symbol", "oi_pct_chg", "close"]],  on="symbol", how="left")
    df = df.merge(pcr_df, on="symbol", how="left")

    # 4. Scoring logic
    def score_row(r):
        score = 0
        if r.get("oi_pct_chg", 0) >= 5:
            score += 1
        if 1.0 <= r.get("pcr", 0) <= 1.6:
            score += 1
        if r.get("avg20vol", 0) > 0 and (r.get("volume", 0) / r["avg20vol"] >= 1.5):
            score += 1
        return score

    df["Score"] = df.apply(score_row, axis=1)

    # 5. Output
    df.sort_values("Score", ascending=False).to_csv(OUTPUT_CSV, index=False, mode='a', header=False)
    print(f"Enriched file exported to {OUTPUT_CSV}")

if __name__ == "__main__":
    main()

Fetching https://nsearchives.nseindia.com/content/fo/BhavCopy_NSE_FO_0_0_0_20251027_F_0000.csv.zip
Fetching https://nsearchives.nseindia.com/content/fo/BhavCopy_NSE_FO_0_0_0_20251024_F_0000.csv.zip
Enriched file exported to enriched_shortlist.csv


In [2]:
import json
import pandas as pd
import requests, zipfile, io, datetime as dt, sys, os
import numpy as np
from io import StringIO
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# -------------------------- USER SETTINGS -----------------------------
ENRICHED_CSV   = "enriched_shortlist.csv"
DATA_STORE_DIR = r"C:\Users\bkaly\Documents\Trading\downloads"
# ----------------------------------------------------------------------

today = dt.datetime.today().date().strftime("%d-%m-%Y")

# Date Change applies here
# today = dt.date(2025, 10, 1).strftime("%d-%m-%Y")  # For testing, use a fixed date

# 1. Load Chartink shortlist
screenersymbols = pd.read_csv(ENRICHED_CSV)
screenersymbols["Date"] = pd.to_datetime(screenersymbols["Date"], format='mixed', dayfirst=True) # Ensure Date is in datetime format
# symbols = screenersymbols[(screenersymbols["Date"] == pd.to_datetime(today, format='%d-%m-%Y')) & (screenersymbols["Score"] >= 2)]
symbols = screenersymbols[(screenersymbols["Date"] == pd.to_datetime(today, format='%d-%m-%Y'))]

# 1.5 Get Option chain data for each symbol
def nsefetch(symbol: str):

    payload = 'https://www.nseindia.com/api/option-chain-equities?symbol='+symbol.replace("&", "%26")

    headers = {
            "accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
            "accept-language": "en-US,en;q=0.9,en-IN;q=0.8,en-GB;q=0.7",
            "cache-control": "max-age=0",
            "priority": "u=0, i",
            "sec-ch-ua": '"Microsoft Edge";v="129", "Not=A?Brand";v="8", "Chromium";v="129"',
            "sec-ch-ua-mobile": "?0",
            "sec-ch-ua-platform": '"Windows"',
            "sec-fetch-dest": "document",
            "sec-fetch-mode": "navigate",
            "sec-fetch-site": "none",
            "sec-fetch-user": "?1",
            "upgrade-insecure-requests": "1",
            "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/129.0.0.0 Safari/537.36 Edg/129.0.0.0"
        }

    try:
        s = requests.Session()
        s.get("https://www.nseindia.com", headers=headers, timeout=10)
        s.get("https://www.nseindia.com/option-chain", headers=headers, timeout=10)
        output = s.get(payload, headers=headers, timeout=10).json()            
    except ValueError:
        output = {}
    return output


# 2. get expirations dates
def _option_expirations_list(identifier: str) : #-> list:    
    # df = pd.read_json(os.path.join(DATA_STORE_DIR, f"{identifier}.json"))
    data = json.dumps(nsefetch(identifier))
    # Wrap the JSON string in StringIO
    json_file_like = StringIO(data)

    df = pd.read_json(json_file_like)

    if df.empty:
        print(f"No data found for {identifier}. Skipping...")
        return [], df

    # print(df)
    expirations_list = list(df['records'].expiryDates)

    # Save the fetched data to a JSON file for future use
    # df.to_json(os.path.join(DATA_STORE_DIR, f"{identifier}.json"), orient='records', date_format='iso')
    return expirations_list, df

# 3. get latest stock price
def _latest_stock_price(identifier:str, df: pd.DataFrame):
    latest_stock_price = df['records'].underlyingValue
    return latest_stock_price

# 4. get option strike price and implied volatility
def _option_strike_price_and_implied_volatility(latest_stock_price, options_dataset: pd.DataFrame):
    
    option_strike_price, implied_volatility = None, None

    try:
        # Filter For Near The Money Option Strike Info
        near_the_money_option = options_dataset.iloc[(options_dataset["CE.strikePrice"]-latest_stock_price).abs().argsort()[:2]].iloc[0]
        

        # # Grab Required Stats
        implied_volatility = near_the_money_option["CE.impliedVolatility"]
        option_strike_price = near_the_money_option["CE.strikePrice"]

        return option_strike_price, implied_volatility
    except KeyError:
        return option_strike_price, implied_volatility

# 5. get option strike price and implied volatility
def _stock_standard_deviation_range(option_strike_price, implied_volatility, option_expiration_date):
    upper_range, lower_range = None, None

    if option_strike_price or implied_volatility != None:
        days_in_year = 252
        option_expiration_datetime = dt.datetime.strptime(option_expiration_date, "%d-%b-%Y").date()
        days_until_expiration = (option_expiration_datetime - dt.date.today()).days

        # days_expo_ratio = days_until_expiration / days_in_year
        days_expo_ratio = 1 / days_in_year
        days_expo_ratio_sqrt = np.sqrt(days_expo_ratio)

        std_dev_price_range_by_expo = option_strike_price * (implied_volatility / 100) * days_expo_ratio_sqrt 
        std_dev_price_range_by_expo = round(std_dev_price_range_by_expo, 2)

        upper_range = round((option_strike_price + std_dev_price_range_by_expo), 2)
        lower_range = round((option_strike_price - std_dev_price_range_by_expo), 2)

    return upper_range, lower_range

minProfit: int = 2000
maxLoss: int = 500

for index, row in symbols.iterrows():
    
    #print(row)    
    identifier = row['symbol']
    lotSize = row['lot size']
    
    print(identifier)

    # 1. Get expirations list and latest stock price
    expirations_list, rawoptionchain = _option_expirations_list(identifier)
    if rawoptionchain.empty or expirations_list == []:
        continue
    latest_stock_price = _latest_stock_price(identifier, rawoptionchain)

    # 2. Get current expiration date
    if not expirations_list:
        print(f"No expirations found for {identifier}. Skipping...")
        continue
    expiration_date = expirations_list[0]  # Use the first expiration date for processing

    # Create Options Dataframe
    options_chain = rawoptionchain['records'].data
    options_dataset = pd.json_normalize(options_chain)

    # 3. Filter options chain data for the current expiration date
    if options_dataset.empty:
        print(f"No options data found for {identifier} on {expiration_date}. Skipping...")
        continue
    currentExpiry_dataset = options_dataset[options_dataset["expiryDate"] == expiration_date]
    
    option_strike_price, implied_volatility = _option_strike_price_and_implied_volatility(latest_stock_price,currentExpiry_dataset)
    upper_range, lower_range = _stock_standard_deviation_range(latest_stock_price, implied_volatility, expiration_date)    

    screenersymbols.loc[(screenersymbols["Date"] == pd.to_datetime(today, format='%d-%m-%Y')) & (screenersymbols["symbol"] == identifier), 'implied_volatility'] = implied_volatility
    screenersymbols.loc[(screenersymbols["Date"] == pd.to_datetime(today, format='%d-%m-%Y')) & (screenersymbols["symbol"] == identifier), 'upper_range'] = upper_range
    screenersymbols.loc[(screenersymbols["Date"] == pd.to_datetime(today, format='%d-%m-%Y')) & (screenersymbols["symbol"] == identifier), 'lower_range'] = lower_range
    screenersymbols.loc[(screenersymbols["Date"] == pd.to_datetime(today, format='%d-%m-%Y')) & (screenersymbols["symbol"] == identifier), 'MinProfit'] = (minProfit / lotSize)
    screenersymbols.loc[(screenersymbols["Date"] == pd.to_datetime(today, format='%d-%m-%Y')) & (screenersymbols["symbol"] == identifier), 'MaxLoss'] = (maxLoss / lotSize)

    # screenersymbols.loc[(screenersymbols["Date"] == pd.to_datetime(today, format='%d-%m-%Y')) & (screenersymbols["Score"] >= 2) & (screenersymbols["symbol"] == identifier), 'implied_volatility'] = implied_volatility
    # screenersymbols.loc[(screenersymbols["Date"] == pd.to_datetime(today, format='%d-%m-%Y')) & (screenersymbols["Score"] >= 2) & (screenersymbols["symbol"] == identifier), 'upper_range'] = upper_range
    # screenersymbols.loc[(screenersymbols["Date"] == pd.to_datetime(today, format='%d-%m-%Y')) & (screenersymbols["Score"] >= 2) & (screenersymbols["symbol"] == identifier), 'lower_range'] = lower_range


screenersymbols.sort_values(["Date","Score"], ascending=False).to_csv(ENRICHED_CSV, index=False)

IOC
No data found for IOC. Skipping...
BPCL
No data found for BPCL. Skipping...
HINDALCO
No data found for HINDALCO. Skipping...
CUMMINSIND
No data found for CUMMINSIND. Skipping...
ASHOKLEY
No data found for ASHOKLEY. Skipping...
KALYANKJIL
No data found for KALYANKJIL. Skipping...
MOTHERSON
No data found for MOTHERSON. Skipping...
BANKINDIA
No data found for BANKINDIA. Skipping...
NYKAA
No data found for NYKAA. Skipping...
TITAGARH
No data found for TITAGARH. Skipping...
RECLTD
No data found for RECLTD. Skipping...
OFSS
No data found for OFSS. Skipping...
UNITDSPR
No data found for UNITDSPR. Skipping...
ETERNAL
No data found for ETERNAL. Skipping...
DRREDDY
No data found for DRREDDY. Skipping...
ICICIBANK
No data found for ICICIBANK. Skipping...
PAYTM
No data found for PAYTM. Skipping...
INDUSINDBK
No data found for INDUSINDBK. Skipping...
BHEL
No data found for BHEL. Skipping...
HINDPETRO
No data found for HINDPETRO. Skipping...
ITC
No data found for ITC. Skipping...
VEDL
No data fo